In [ ]:
import pandas as pd
import random
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import json
import os
import torch

# --- 1. TinyLlama Generator ---
class TinyLlamaGenerator:
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        print(f"Loading TinyLlama model: {model_name}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)

            # --- FIX for "right-padding detected" warning ---
            self.tokenizer.padding_side = 'left'
            print("Tokenizer padding_side set to 'left' for correct generation results.")

            # Add a pad_token if it's missing (common for some models, needed for batching)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                print("Tokenizer pad_token set to eos_token for batching.")

            # --- Memory Optimization: Load model in half-precision ---
            if torch.cuda.is_available():
                if torch.cuda.get_device_capability()[0] >= 8: # Ampere or newer for bfloat16
                    torch_dtype = torch.bfloat16
                    print("Using torch.bfloat16 for model loading.")
                else:
                    torch_dtype = torch.float16
                    print("Using torch.float16 for model loading.")
            else:
                torch_dtype = None # No specific dtype for CPU, defaults to float32
                print("Running on CPU. Model will be loaded in default precision (float32).")

            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch_dtype,
                low_cpu_mem_usage=True
            )

            device = 0 if torch.cuda.is_available() else -1
            self.generator = pipeline(
                "text-generation",
                model=self.model,
                tokenizer=self.tokenizer,
                device=device,
                max_new_tokens=150,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id, # Use pad_token_id for pipeline
                return_full_text=False
            )
            print("TinyLlama model loaded successfully.")
            self._is_dummy = False
        except Exception as e:
            print(f"Error loading TinyLlama model: {e}")
            print("Please ensure the model is downloaded and sufficient resources are available.")
            print("Falling back to a dummy generator for demonstration purposes.")
            self._is_dummy = True
            class DummyTinyLlamaGenerator:
                def generate_responses(self, prompts_list, temperatures):
                    results = []
                    for i, prompt in enumerate(prompts_list):
                        temp = temperatures[i] if len(temperatures) > i else 0.7
                        if temp < 0.5:
                            results.append(f"This is a precise and high-quality response to '{prompt}'. (temp={temp:.1f})")
                        elif temp < 0.8:
                            results.append(f"Here's a decent response to '{prompt}'. (temp={temp:.1f})")
                        else:
                            results.append(f"This response to '{prompt}' is a bit off-topic. (temp={temp:.1f})")
                    return results
            self.generator = DummyTinyLlamaGenerator()
            self.tokenizer = None
            self.model = None

    def _generate_for_single_temperature_batch(self, prompts_list, temperature, max_new_tokens=150):
        if self._is_dummy:
            # Dummy generator can simulate different temperatures per prompt if needed for its dummy logic
            dummy_temperatures = [temperature] * len(prompts_list)
            return self.generator.generate_responses(prompts_list, dummy_temperatures, max_new_tokens)

        chat_prompts = []
        for prompt in prompts_list:
            messages = [{"role": "user", "content": prompt}]
            chat_prompts.append(self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        # Pass all templated prompts to the generator at once
        outputs = self.generator(
            chat_prompts,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            batch_size=len(chat_prompts) # Explicitly tell pipeline to batch internally
        )

        response_texts = []
        for output in outputs:
            # Access the dictionary within the inner list
            response_text = output[0]['generated_text'].strip()
            if "[/INST]" in response_text:
                response_text = response_text.split("[/INST]")[-1].strip()
            response_texts.append(response_text)
        return response_texts


# --- 2. Text Quality Classifier ---
class TextQualityClassifier:
    def __init__(self):
        print("TextQualityClassifier initialized (placeholder).")
        pass

    def predict_quality_score(self, text: str) -> float:
        if "bad" in text.lower() or "irrelevant" in text.lower() or "nonsensical" in text.lower() or "a bit off" in text.lower():
            return random.uniform(0.0, 0.3)
        elif "good" in text.lower() or "excellent" in text.lower() or "clear" in text.lower() or "precise" in text.lower():
            return random.uniform(0.7, 1.0)
        else:
            return random.uniform(0.3, 0.9)


# --- 3. Main Logic for Generating Response Pairs ---

def generate_rlhf_response_pairs_two_responses(prompts_batch, tinyllama_generator, quality_classifier):
    """
    Generates chosen/rejected response pairs for a batch of prompts by generating two sets
    of responses (one with low temp, one with high temp) for the entire batch.
    """
    response_pairs_batch = []

    # Generate all Response1 (low temp) for the current batch
    print(f"  Generating responses with temperature=0.5 for {len(prompts_batch)} prompts...")
    responses1_batch = tinyllama_generator._generate_for_single_temperature_batch(prompts_batch, temperature=0.5, max_new_tokens=150)

    # Generate all Response2 (high temp) for the current batch
    print(f"  Generating responses with temperature=0.9 for {len(prompts_batch)} prompts...")
    responses2_batch = tinyllama_generator._generate_for_single_temperature_batch(prompts_batch, temperature=0.9, max_new_tokens=150)

    # Now, process each prompt's two responses and classify
    for i, prompt in enumerate(prompts_batch):
        response1 = responses1_batch[i]
        response2 = responses2_batch[i]

        score1 = quality_classifier.predict_quality_score(response1)
        score2 = quality_classifier.predict_quality_score(response2)

        if score1 >= score2:
            chosen_response = response1
            rejected_response = response2
            chosen_score = score1
            rejected_score = score2
        else:
            chosen_response = response2
            rejected_response = response1
            chosen_score = score2
            rejected_score = score1

        if chosen_response == rejected_response:
            print(f"Warning: Chosen and rejected responses are identical for prompt: '{prompt[:70]}...'. Attempting to generate alternative.")
            # Fallback for identical responses - generate one more with max temperature
            fallback_response = tinyllama_generator._generate_for_single_temperature_batch([prompt], temperature=1.0, max_new_tokens=150)[0]
            fallback_score = quality_classifier.predict_quality_score(fallback_response)

            if fallback_score < chosen_score and fallback_response != chosen_response:
                rejected_response = fallback_response
                rejected_score = fallback_score
            else:
                print("  Could not find a distinct rejected response even with fallback. Skipping prompt.")
                continue

        response_pairs_batch.append({
            "prompt": prompt,
            "chosen": chosen_response,
            "rejected": rejected_response
        })

    return response_pairs_batch

# --- Example Usage ---
if __name__ == "__main__":
    # --- Configuration ---
    prompts_csv_path = "prompts.csv"
    output_jsonl_path = "rlhf_response_pairs.jsonl" # Changed to .jsonl for appending
    batch_size = 100

    # --- 1. Load Prompts from CSV ---
    if not os.path.exists(prompts_csv_path):
        print(f"Error: '{prompts_csv_path}' not found.")
        print("Please ensure your prompts CSV file is in the same directory or provide the full path.")
        exit("Exiting: prompts.csv not found.")

    print(f"Loading prompts from {prompts_csv_path}...")
    try:
        df_prompts = pd.read_csv(prompts_csv_path, header=None)
        all_prompts = df_prompts.iloc[:, 0].tolist()
        all_prompts = [p for p in all_prompts if isinstance(p, str) and p.strip()]

        if not all_prompts:
            raise ValueError("No valid prompts found in the CSV after cleaning.")

        print(f"Loaded {len(all_prompts)} valid prompts.")
    except Exception as e:
        print(f"Error reading prompts from CSV: {e}")
        print("Please check your CSV file format and content. Exiting.")
        exit(f"Exiting due to CSV reading error: {e}")

    # --- 2. Initialize Generator and Classifier ---
    tinyllama_gen = TinyLlamaGenerator()
    quality_classifier = TextQualityClassifier()

    # --- 3. Generate Response Pairs in Batches ---
    total_prompts = len(all_prompts)
    processed_count = 0

    # Open the output file in append mode. If you want to start fresh, delete the .jsonl file first.
    with open(output_jsonl_path, "a", encoding="utf-8") as f_out:
        for i in range(0, total_prompts, batch_size):
            batch_prompts = all_prompts[i : i + batch_size]
            print(f"\nProcessing batch {i // batch_size + 1} (Prompts {i+1} to {min(i + batch_size, total_prompts)})...")

            # Generate pairs for the current batch
            current_batch_pairs = generate_rlhf_response_pairs_two_responses(
                prompts_batch=batch_prompts,
                tinyllama_generator=tinyllama_gen,
                quality_classifier=quality_classifier
            )

            # Write pairs from the current batch to the JSONL file
            for pair in current_batch_pairs:
                f_out.write(json.dumps(pair, ensure_ascii=False) + "\n")
            f_out.flush() # Ensure data is written to disk

            processed_count += len(current_batch_pairs)
            print(f"Batch processed. Total processed: {processed_count}/{total_prompts}")

            # --- Memory Management: Clear CUDA cache after each batch ---
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                print("CUDA cache cleared.")

print("\n--- Generation Complete ---")
print(f"Generated {processed_count} response pairs and saved to {output_jsonl_path}")

Loading prompts from prompts.csv...
Loaded 2529 valid prompts.
Loading TinyLlama model: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer padding_side set to 'left' for correct generation results.
Using torch.float16 for model loading.


Device set to use cuda:0


TinyLlama model loaded successfully.
TextQualityClassifier initialized (placeholder).

Processing batch 1 (Prompts 1 to 100)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 100/2529
CUDA cache cleared.

Processing batch 2 (Prompts 101 to 200)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 200/2529
CUDA cache cleared.

Processing batch 3 (Prompts 201 to 300)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 300/2529
CUDA cache cleared.

Processing batch 4 (Prompts 301 to 400)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 400/2529
CUDA

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Batch processed. Total processed: 500/2529
CUDA cache cleared.

Processing batch 6 (Prompts 501 to 600)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 600/2529
CUDA cache cleared.

Processing batch 7 (Prompts 601 to 700)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 700/2529
CUDA cache cleared.

Processing batch 8 (Prompts 701 to 800)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 800/2529
CUDA cache cleared.

Processing batch 9 (Prompts 801 to 900)...
  Generating responses with temperature=0.5 for 100 prompts...
  Generating responses with temperature=0.9 for 100 prompts...
Batch processed. Total processed: 900/2529
CUDA cache cleared.

Proc